In [5]:
import os
import time
import hashlib
import pandas as pd
from selenium.webdriver.common.by import By
import undetected_chromedriver as uc

In [12]:
# Leer el token desde el archivo externo
ruta_token = "auth.tk"
try:
    with open(ruta_token, "r", encoding="utf-8") as archivo:
        mi_auth_token = archivo.read().strip()
except FileNotFoundError:
    print(f"ERROR: No se encontró el archivo '{ruta_token}'.")
    print("Crea el archivo en la misma carpeta del script, pega tu token dentro y vuelve a ejecutar.")
    driver.quit()
    exit()

In [11]:
def seudonimizar_usuario(username):
    """Aplica un hash SHA-256 al nombre de usuario para cumplir con la Ley 21.719."""
    if not username: return None
    return hashlib.sha256(username.encode('utf-8')).hexdigest()

def extraer_comentarios_x(url, periodo, driver, limite=300):
    driver.get(url)
    print(f"Cargando {periodo}...")
    time.sleep(6) # Pausa inicial para carga de DOM
    
    comentarios_extraidos = []
    ids_vistos = set()
    intentos_sin_nuevos = 0 # Contador para evitar bucles infinitos
    
    # Bucle de scroll infinito para cargar comentarios dinámicos
    while len(comentarios_extraidos) < limite:
        
        # 1. Salvaguarda: Si el script hizo un clic accidental y salió del post, forzar el regreso
        url_base = url.split('?')[0] # Ignorar parámetros extra
        if not driver.current_url.startswith(url_base):
            print("El navegador se desvió de la página principal. Regresando...")
            driver.get(url)
            time.sleep(5)
            
        cantidad_anterior = len(comentarios_extraidos)
        articulos = driver.find_elements(By.XPATH, '//article[@data-testid="tweet"]')
        
        for articulo in articulos:
            try:
                texto_elem = articulo.find_elements(By.XPATH, './/div[@data-testid="tweetText"]')
                texto = texto_elem[0].text if texto_elem else None
                
                user_elem = articulo.find_elements(By.XPATH, './/div[@dir="ltr"]/span')
                usuario_raw = user_elem[0].text if user_elem else "usuario_desconocido"
                
                time_elem = articulo.find_elements(By.XPATH, './/time')
                fecha_hora = time_elem[0].get_attribute('datetime') if time_elem else None
                
                comentario_id = f"{usuario_raw}-{fecha_hora}"
                
                if texto and comentario_id not in ids_vistos:
                    ids_vistos.add(comentario_id)
                    
                    comentarios_extraidos.append({
                        "id_texto": len(ids_vistos),
                        "fecha_hora": fecha_hora,
                        "id_usuario_hash": seudonimizar_usuario(usuario_raw),
                        "texto": texto,
                        "consulta_origen": url,
                        "periodo": periodo
                    })
            except Exception:
                continue 
                
        if len(comentarios_extraidos) >= limite:
            break
            
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3) 
        
        # 2. Validación y clic en "Mostrar más"
        if len(comentarios_extraidos) == cantidad_anterior:
            intentos_sin_nuevos += 1
            
            try:
                # Selector más estricto: buscar solo dentro de contenedores que sean botones reales
                botones = driver.find_elements(By.XPATH, "//div[@role='button']//span[contains(text(), 'Mostrar') or contains(text(), 'Show')]")
                for btn in botones:
                    if btn.is_displayed():
                        driver.execute_script("arguments[0].click();", btn)
                        print("Botón de 'Mostrar' presionado...")
                        time.sleep(3)
                        # ¡ATENCIÓN! No reiniciamos el contador aquí. 
                        # Solo se reiniciará en la próxima vuelta si el clic trajo datos reales.
                        break
            except Exception:
                pass
                
            # Salida de emergencia estricta tras 4 intentos fallidos reales
            if intentos_sin_nuevos >= 4:
                print(f"Fondo de la página alcanzado. Obtenidos: {len(comentarios_extraidos)} de {limite}.")
                break
        else:
            # Aquí sí reiniciamos el contador porque la cantidad de extracción subió
            intentos_sin_nuevos = 0 
            print(f"Extrayendo... {len(comentarios_extraidos)} / {limite}")
            
    return comentarios_extraidos[:limite]

# ==========================================
# 1. Configurar el WebDriver Indetectable
# ==========================================
ruta_perfil_local = os.path.abspath(os.path.join(os.getcwd(), "perfil_selenium_uc"))
if not os.path.exists(ruta_perfil_local):
    os.makedirs(ruta_perfil_local)

opciones = uc.ChromeOptions()
opciones.add_argument(f"--user-data-dir={ruta_perfil_local}") 
opciones.add_argument("--disable-dev-shm-usage")

# Iniciar el navegador. Si sigues con el error de versión, usa: uc.Chrome(options=opciones, version_main=155)
driver = uc.Chrome(options=opciones, version_main=155)

# ==========================================
# 2. Login por Inyección de Cookie
# ==========================================
print("Inyectando sesión de X...")
driver.get("https://x.com/robots.txt")
time.sleep(2)


driver.add_cookie({
    'name': 'auth_token',
    'value': mi_auth_token,
    'domain': '.x.com'
})

# Refrescar la página principal
driver.get("https://x.com/")
time.sleep(5)
print("Sesión inyectada con éxito. Iniciando extracción...")

# ==========================================
# 3. Recolección de Datos
# ==========================================
links = [
    {"url": "https://x.com/NintendoLatam/status/2064362016349757477", "periodo": "Antes (Expectativa / Anuncio)"},
    {"url": "https://x.com/NintendoLatam/status/2097333921260208303", "periodo": "Después (Tráiler Gameplay)"}
]

datos_totales = []
for link in links:
    datos = extraer_comentarios_x(link["url"], link["periodo"], driver, limite=300)
    datos_totales.extend(datos)

driver.quit()

# ==========================================
# 4. Estructurar en Pandas y exportar
# ==========================================
df_comentarios = pd.DataFrame(datos_totales)

# Prevenir error si la muestra solicitada es mayor a los datos obtenidos
muestra_size = min(50, len(df_comentarios))
df_muestra_ruido = df_comentarios.sample(n=muestra_size, random_state=42) if muestra_size > 0 else pd.DataFrame()

df_comentarios.to_csv("corpus_ocarina_remake.csv", index=False, encoding="utf-8")
if not df_muestra_ruido.empty:
    df_muestra_ruido.to_csv("muestra_ruido_manual.csv", index=False, encoding="utf-8")

print("\nExtracción completada. Primeras 5 filas:")
print(df_comentarios.head())

Inyectando sesión de X...
Sesión inyectada con éxito. Iniciando extracción...
Cargando Antes (Expectativa / Anuncio)...


KeyboardInterrupt: 